<a href="https://colab.research.google.com/github/desouki76/Ahmed/blob/main/TwitterFeedback_%2Bve_ve_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Binary Text Classfcaiton [+ve / - Ve for twittes] fille have 5000 +Ve Twiites and file have -Ve Twittes

# Libraires

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk


In [5]:
from numpy import random
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
#from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
#from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
#from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense, SimpleRNN
from sklearn.preprocessing import LabelEncoder

In [37]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tap')
nltk.download('omw-1.4')
nltk.download('twitter_samples')

import nltk

nltk.download('punkt')
nltk.download('punkt_tab')   # <— this is the new one in NLTK 3.9+
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Error loading punkt_tap: Package 'punkt_tap' not found in
[nltk_data]     index
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package twitter_samples to /root/nltk_data...
[nltk_data]   Package twitter_samples is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwo

True

In [7]:
stop_words = stopwords.words('english')  # i call stope of words of english langauge
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

In [8]:
positive_tweets = nltk.corpus.twitter_samples.strings('positive_tweets.json')
# we call class of 5000  postive twittes
negative_tweets = nltk.corpus.twitter_samples.strings('negative_tweets.json')
# we call class of 5000 negtive twittes and we put it in the varable

In [73]:
pd.DataFrame(positive_tweets).info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       5000 non-null   object
dtypes: object(1)
memory usage: 39.2+ KB


In [72]:
pd.DataFrame(negative_tweets).info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       5000 non-null   object
dtypes: object(1)
memory usage: 39.2+ KB


# Fucaiton to clean twittes

In [71]:
pd.DataFrame(positive_tweets).head()


,0
0,#FollowFriday @France_Inte @PKuchly57 @Milipol_Paris for being top engaged members in my community this week :)
1,@Lamb2ja Hey James! How odd :/ Please call our Contact Centre on 02392441234 and we will be able to assist you :) Many thanks!
2,@DespiteOfficial we had a listen last night :) As You Bleed is an amazing track. When are you in Scotland?!
3,@97sides CONGRATS :)
4,yeaaaah yippppy!!! my accnt verified rqst has succeed got a blue tick mark on my fb profile :) in 15 days


In [52]:
#function to clean twittes
def clean_text(tweet):
    tweet=re.sub(r'http\S+','',tweet) #Plass, @mohamed remover https \S+ any no white space after http
    tweet=re.sub(r'@[A-Za-z0-9_]+','',tweet) #
    tweet=re.sub(r'[^\w\s]','',tweet)
    tweet=re.sub(r'\d+','',tweet)
    tweet=re.sub(r'\s+',' ',tweet)
    tweet=re.sub(r'\s+[a-zA-Z]\s+',' ',tweet)
    tweet=re.sub(r'^[a-zA-Z]\s+',' ',tweet)
    tweet=re.sub(r'\s$',' ',tweet)

    return tweet

In [88]:
# creat function to clean tweets and pre preoss as NLP
def preprocess_tweet(tweets):
    clean_tweets=[]
    for tweet in tweets:
      clean_tweet = clean_text(tweet)
      clean_tweet = nltk.word_tokenize(clean_tweet)
      #tweet_tokens=word_tokenize(tweet)
      #clean_tweet=[word for word in clean_tweet if word not in stop_words]
      #clean_tweet=[lemmatizer.lemmatize(word) for word in clean_tweet]

      #apply "lower case" on all tweets if the word.lower() not in stop words
      clean_tweet =[word.lower() for word in clean_tweet if word.lower() not in stop_words]
      #Apply stemmer رد الكلمه الى اصلها
      clean_tweet=[stemmer.stem(word) for word in clean_tweet]
      clean_tweet=[lemmatizer.lemmatize(word) for word in clean_tweet]
      clean_tweet = [word for word in clean_tweet if len(word) >= 3]
      #delet all words that is 2 letters only
      clean_tweets.append(clean_tweet)
    return clean_tweet



In [89]:
#Apply the prvouis fucation pre process (clean, stop words, lower case, limtizer Port stemmer)
positive_tweets_1=preprocess_tweet(positive_tweets)
negative_tweets_1=preprocess_tweet(negative_tweets)

now tweets are clean lower case limitzed Portstemmed toknized to words


In [62]:

# import sys
# sys.setrecursionlimit(10000)  # allow deeper printing if needed

# import numpy as np
# np.set_printoptions(threshold=np.inf)  # show full arrays

# pd.set_option('display.max_colwidth', None)
# pd.set_option('display.max_rows', None)

In [84]:
positive_tweets_1

['could', 'say', 'egg', 'face']

In [85]:
for tweet in positive_tweets_1[:1000]:
    print(tweet)

could
say
egg
face


In [90]:
print(positive_tweets_1[110])

IndexError: list index out of range

In [91]:
pd.DataFrame(positive_tweets_1).info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       4 non-null      object
dtypes: object(1)
memory usage: 164.0+ bytes


In [92]:
pd.DataFrame(negative_tweets_1).info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       5 non-null      object
dtypes: object(1)
memory usage: 172.0+ bytes
